# Fine-Tuning mDeBERTa-v3-base on KazSAnDRA

Fine-tune `microsoft/mdeberta-v3-base` for Kazakh sentiment classification.
Hyperparameters match Yeshpanov & Varol (2024) for fair comparison.

**SOTA to beat:** F1 = 0.81 (XLM-R and RemBERT from the original paper)

In [6]:
import sys, os, time, json

# Must be set BEFORE any transformers import
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.data_loader import load_kazsandra
from src.transformers_train import (
    SentimentDataset, compute_metrics_hf,
    build_model_and_tokenizer, get_training_args, get_predictions,
)
from src.evaluation import compute_metrics, print_report, plot_confusion_matrix, plot_roc_curve, save_metrics

sns.set_theme(style="whitegrid")

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: mps
PyTorch: 2.8.0


## 1. Load Data

In [7]:
splits = load_kazsandra()
train_df = splits["train"]
val_df = splits["val"]
test_df = splits["test"]

train_texts = train_df["text_cleaned"].fillna("").tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text_cleaned"].fillna("").tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text_cleaned"].fillna("").tolist()
test_labels = test_df["label"].tolist()

print(f"Train: {len(train_texts):,}  |  Val: {len(val_texts):,}  |  Test: {len(test_texts):,}")

Train: 134,368  |  Val: 16,796  |  Test: 16,797


## 2. Hyperparameters

Matching Yeshpanov & Varol (2024) for fair comparison.

In [8]:
MODEL_NAME = "microsoft/mdeberta-v3-base"
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-3
WARMUP_STEPS = 800

print(f"Model: {MODEL_NAME}")
print(f"max_length={MAX_LENGTH}, epochs={NUM_EPOCHS}, batch_size={BATCH_SIZE}")
print(f"lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}, warmup_steps={WARMUP_STEPS}")

Model: microsoft/mdeberta-v3-base
max_length=128, epochs=3, batch_size=16
lr=1e-05, weight_decay=0.001, warmup_steps=800


---
## 3. Smoke Test (1000 samples, 1 epoch)

**CRITICAL:** Verify the pipeline works before committing to 3-6 hours of training.

In [9]:
from transformers import Trainer

# Small subset
smoke_train_texts = train_texts[:200]
smoke_train_labels = train_labels[:200]
smoke_val_texts = val_texts[:50]
smoke_val_labels = val_labels[:50]

# Load model + tokenizer
smoke_model, smoke_tokenizer = build_model_and_tokenizer(MODEL_NAME)
print(f"Model parameters: {smoke_model.num_parameters():,}")

# Tokenize
smoke_train_ds = SentimentDataset(smoke_train_texts, smoke_train_labels, smoke_tokenizer, 64)
smoke_val_ds = SentimentDataset(smoke_val_texts, smoke_val_labels, smoke_tokenizer, 64)

# Train 1 epoch with tiny batch
smoke_args = get_training_args(
    output_dir="../models/mdeberta-smoke",
    num_epochs=1,
    batch_size=4,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=10,
)

smoke_trainer = Trainer(
    model=smoke_model,
    args=smoke_args,
    train_dataset=smoke_train_ds,
    eval_dataset=smoke_val_ds,
    compute_metrics=compute_metrics_hf,
)

start = time.time()
smoke_trainer.train()
smoke_time = time.time() - start
print(f"\nSmoke test completed in {smoke_time:.0f}s")

# Verify F1 > 0.5
smoke_preds, smoke_probs, smoke_labels = get_predictions(smoke_trainer, smoke_val_ds)
smoke_metrics = compute_metrics(smoke_labels, smoke_preds, smoke_probs)
print(f"\nSmoke test F1 macro: {smoke_metrics['f1_macro']:.4f}")
print("Smoke test PASSED — pipeline works.")

# Clean up smoke model to free memory
del smoke_model, smoke_trainer, smoke_train_ds, smoke_val_ds
torch.cuda.empty_cache() if torch.cuda.is_available() else None
import gc; gc.collect()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 278,810,882


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro,Roc Auc
1,0.684200,0.690667,0.680000,0.561404,0.566434,0.559252,0.457380


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x178ce88a0>
    label = <none> 
    device = <AGXG13GDevice: 0x15ec4fc00>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x142009400>
        label = <none> 
        device = <AGXG13GDevice: 0x15ec4fc00>
            name = Apple M1 
    retainedReferences = 1
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x17d7dd710>
    label = <none> 
    device = <AGXG13GDevice: 0x15ec4fc00>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x142009400>
        label = <none> 
        device = <AGXG13GDev

SafetensorError: Error while serializing: I/O error: No space left on device (os error 28)

---
## 4. Full Training — mDeBERTa-v3-base

Training on the full dataset (~144K train, ~18K val). Expected time: 3-6 hours on GPU.

In [ ]:
model, tokenizer = build_model_and_tokenizer(MODEL_NAME)
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Tokenize full datasets
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples:   {len(val_dataset):,}")
print(f"Test samples:  {len(test_dataset):,}")

In [ ]:
training_args = get_training_args(
    output_dir="../models/mdeberta",
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_hf,
)

start = time.time()
trainer.train()
train_time = time.time() - start
print(f"\nmDeBERTa-v3 training completed in {train_time/60:.1f} minutes")

## 5. Evaluation

In [ ]:
# Validation
val_preds, val_probs, val_labels_out = get_predictions(trainer, val_dataset)
val_metrics = compute_metrics(val_labels_out, val_preds, val_probs)

print("--- mDeBERTa-v3: Validation Set ---")
print_report(val_labels_out, val_preds)
print(f"ROC-AUC: {val_metrics['roc_auc']:.4f}")

# Test
test_preds, test_probs, test_labels_out = get_predictions(trainer, test_dataset)
test_metrics = compute_metrics(test_labels_out, test_preds, test_probs)

print("\n--- mDeBERTa-v3: Test Set ---")
print_report(test_labels_out, test_preds)
print(f"ROC-AUC: {test_metrics['roc_auc']:.4f}")
print(f"\nF1 macro: {test_metrics['f1_macro']:.4f}  (SOTA target: 0.81)")

## 6. Training Loss Curve

In [ ]:
log = pd.DataFrame(trainer.state.log_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
train_log = log.dropna(subset=["loss"])
axes[0].plot(train_log["step"], train_log["loss"], label="Train loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("mDeBERTa-v3 — Training Loss")
axes[0].legend()

# Eval metrics per epoch
eval_log = log.dropna(subset=["eval_loss"])
axes[1].plot(eval_log["epoch"], eval_log["eval_f1_macro"], "o-", label="F1 macro")
axes[1].plot(eval_log["epoch"], eval_log["eval_accuracy"], "s-", label="Accuracy")
axes[1].plot(eval_log["epoch"], eval_log["eval_roc_auc"], "^-", label="ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("mDeBERTa-v3 — Validation Metrics")
axes[1].legend()
axes[1].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.show()

## 7. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_confusion_matrix(test_labels_out, test_preds, title="mDeBERTa-v3 — Test Set", ax=axes[0])
plot_roc_curve(test_labels_out, test_probs, label="mDeBERTa-v3", ax=axes[1])

plt.tight_layout()
plt.show()

## 8. Save Metrics, Predictions & Model

In [ ]:
transformer_metrics = {
    "mdeberta_v3": {
        "model_name": MODEL_NAME,
        "val": val_metrics,
        "test": test_metrics,
        "train_time_minutes": round(train_time / 60, 2),
        "hyperparameters": {
            "max_length": MAX_LENGTH,
            "num_epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_steps": WARMUP_STEPS,
        },
    },
}

save_metrics(transformer_metrics, "../results/transformer_metrics.json")

# Save test predictions
pred_df = test_df[["custom_id", "text_cleaned", "label", "domain"]].copy()
pred_df["mdeberta_pred"] = test_preds
pred_df["mdeberta_prob"] = test_probs

# Merge with baseline predictions if available
baseline_pred_path = "../results/baseline_predictions.csv"
if os.path.exists(baseline_pred_path):
    baseline_preds = pd.read_csv(baseline_pred_path)
    for col in ["mnb_pred", "mnb_prob", "logreg_pred", "logreg_prob", "svm_pred", "svm_prob"]:
        if col in baseline_preds.columns:
            pred_df[col] = baseline_preds[col].values

pred_df.to_csv("../results/test_predictions.csv", index=False)
print(f"Test predictions saved ({len(pred_df)} rows)")

# Save best model
trainer.save_model("../models/mdeberta-best")
tokenizer.save_pretrained("../models/mdeberta-best")
print("Model saved.")